In [0]:

# GOLD LAYER - STEP 1
# Read cleaned Silver tables

silver_customers = spark.table("silver_customers")
silver_events = spark.table("silver_events")

print("Silver customers:", silver_customers.count())
print("Silver events:", silver_events.count())

display(silver_customers.limit(5))
display(silver_events.limit(5))


Silver customers: 15
Silver events: 30


customer_id,email,signup_timestamp,country,membership_tier,ingested_at,signup_datetime
1003,charlie.brown@example.com,2024-02-18T14:05:00.000Z,Canada,Bronze,2026-11-01T03:14:11.568Z,2024-02-18T14:05:00.000Z
1015,oliver.white@example.com,2024-09-02T08:20:00.000Z,Australia,Bronze,2026-11-01T03:14:11.568Z,2024-09-02T08:20:00.000Z
1002,bob.smith@example.com,2024-02-02T11:15:00.000Z,USA,Silver,2026-11-01T03:14:11.568Z,2024-02-02T11:15:00.000Z
1007,george.miller@example.com,2024-04-21T12:30:00.000Z,Australia,Bronze,2026-11-01T03:14:11.568Z,2024-04-21T12:30:00.000Z
1008,hana.chen@example.com,2024-05-03T13:10:00.000Z,Canada,Silver,2026-11-01T03:14:11.568Z,2024-05-03T13:10:00.000Z


event_id,customer_id,timestamp,event_type,transaction_amount,ingested_at,event_timestamp
5003,1003,2024-09-02T09:15:00.000Z,view,0.00,2026-11-01T03:14:17.817Z,2024-09-02T09:15:00.000Z
5001,1001,2024-09-01T10:00:00.000Z,purchase,129.99,2026-11-01T03:14:17.817Z,2024-09-01T10:00:00.000Z
5006,1006,2024-09-03T16:10:00.000Z,purchase,310.25,2026-11-01T03:14:17.817Z,2024-09-03T16:10:00.000Z
5007,1007,2024-09-04T08:50:00.000Z,cart,0.00,2026-11-01T03:14:17.817Z,2024-09-04T08:50:00.000Z
5016,1001,2024-09-09T12:00:00.000Z,purchase,75.00,2026-11-01T03:14:17.817Z,2024-09-09T12:00:00.000Z


In [0]:

# GOLD LAYER - STEP 2
# Join events with customer information

from pyspark.sql.functions import (
    col,
    to_date,
    when,
    sum,
    count,
    countDistinct,
    avg,
    round,
    coalesce,
    lit
)

gold_base = (
    silver_events.alias("e")
    .join(
        silver_customers.alias("c"),
        col("e.customer_id") == col("c.customer_id"),
        "left"
    )
    .select(
        col("e.event_id"),
        col("e.customer_id"),
        col("e.event_type"),
        col("e.event_timestamp"),
        to_date(col("e.event_timestamp")).alias("event_date"),
        col("e.transaction_amount"),
        col("c.country"),
        col("c.membership_tier")
    )
)

display(gold_base)


event_id,customer_id,event_type,event_timestamp,event_date,transaction_amount,country,membership_tier
5003,1003,view,2024-09-02T09:15:00.000Z,2024-09-02,0.00,Canada,Bronze
5001,1001,purchase,2024-09-01T10:00:00.000Z,2024-09-01,129.99,USA,Gold
5006,1006,purchase,2024-09-03T16:10:00.000Z,2024-09-03,310.25,India,Gold
5007,1007,cart,2024-09-04T08:50:00.000Z,2024-09-04,0.00,Australia,Bronze
5016,1001,purchase,2024-09-09T12:00:00.000Z,2024-09-09,75.00,USA,Gold
5022,1007,purchase,2024-09-13T14:15:00.000Z,2024-09-13,145.75,Australia,Bronze
5023,1008,purchase,2024-09-14T11:50:00.000Z,2024-09-14,89.99,Canada,Silver
5004,1004,purchase,2024-09-02T13:45:00.000Z,2024-09-02,250.00,UK,Gold
5017,1002,purchase,2024-09-10T15:45:00.000Z,2024-09-10,110.00,USA,Silver
5029,1014,purchase,2024-09-17T10:20:00.000Z,2024-09-17,155.00,India,Gold


In [0]:

# GOLD LAYER - STEP 3
# Daily sales summary

gold_daily_sales = (
    gold_base
    .groupBy("event_date")
    .agg(
        count("event_id").alias("total_events"),

        countDistinct(
            when(col("event_type") == "purchase", col("event_id"))
        ).alias("total_purchases"),

        countDistinct("customer_id").alias("unique_customers"),

        round(
            sum(
                when(
                    col("event_type") == "purchase",
                    col("transaction_amount")
                ).otherwise(lit(0))
            ), 2
        ).alias("total_revenue"),

        round(
            avg(
                when(
                    col("event_type") == "purchase",
                    col("transaction_amount")
                )
            ), 2
        ).alias("average_purchase_value")
    )
    .orderBy("event_date")
)

display(gold_daily_sales)


event_date,total_events,total_purchases,unique_customers,total_revenue,average_purchase_value
2024-09-01,2,2,2,179.49,89.75
2024-09-02,2,1,2,250.00,250.00
2024-09-03,2,2,2,390.24,195.12
2024-09-04,2,1,2,65.75,65.75
2024-09-05,2,1,2,499.99,499.99
2024-09-06,2,2,2,288.95,144.48
2024-09-07,2,2,2,365.00,182.50
2024-09-08,1,1,1,39.99,39.99
2024-09-09,1,1,1,75.00,75.00
2024-09-10,1,1,1,110.00,110.00


In [0]:

# GOLD LAYER - STEP 4
# Country-wise revenue

gold_country_sales = (
    gold_base
    .groupBy("country")
    .agg(
        countDistinct("customer_id").alias("total_customers"),

        countDistinct(
            when(col("event_type") == "purchase", col("event_id"))
        ).alias("total_purchases"),

        round(
            sum(
                when(
                    col("event_type") == "purchase",
                    col("transaction_amount")
                ).otherwise(lit(0))
            ), 2
        ).alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

display(gold_country_sales)


country,total_customers,total_purchases,total_revenue
USA,4,8,1328.98
India,3,6,860.74
UK,2,3,565.25
Spain,1,2,519.95
Australia,2,3,395.74
Canada,2,3,215.73
South Korea,1,2,149.00


In [0]:

# GOLD LAYER - STEP 5
# Membership-tier sales performance

gold_membership_sales = (
    gold_base
    .groupBy("membership_tier")
    .agg(
        countDistinct("customer_id").alias("customers_with_events"),

        countDistinct(
            when(col("event_type") == "purchase", col("event_id"))
        ).alias("total_purchases"),

        round(
            sum(
                when(
                    col("event_type") == "purchase",
                    col("transaction_amount")
                ).otherwise(lit(0))
            ), 2
        ).alias("total_revenue"),

        round(
            avg(
                when(
                    col("event_type") == "purchase",
                    col("transaction_amount")
                )
            ), 2
        ).alias("average_purchase_value")
    )
    .orderBy(col("total_revenue").desc())
)

display(gold_membership_sales)


membership_tier,customers_with_events,total_purchases,total_revenue,average_purchase_value
Gold,6,12,2615.18,217.93
Silver,5,9,815.48,90.61
Bronze,4,6,604.73,100.79


In [0]:

# GOLD LAYER - STEP 6
# Overall business KPIs

gold_overall_kpis = (
    gold_base
    .agg(
        countDistinct("customer_id").alias("unique_customers"),

        count("event_id").alias("total_events"),

        countDistinct(
            when(col("event_type") == "purchase", col("event_id"))
        ).alias("total_purchases"),

        round(
            sum(
                when(
                    col("event_type") == "purchase",
                    col("transaction_amount")
                ).otherwise(lit(0))
            ), 2
        ).alias("total_revenue"),

        round(
            avg(
                when(
                    col("event_type") == "purchase",
                    col("transaction_amount")
                )
            ), 2
        ).alias("average_purchase_value")
    )
)

display(gold_overall_kpis)


unique_customers,total_events,total_purchases,total_revenue,average_purchase_value
15,30,27,4035.39,149.46


In [0]:

# GOLD LAYER - STEP 7
# Save Gold Delta tables

gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_sales_summary")

gold_country_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_country_sales")

gold_membership_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_membership_sales")

gold_overall_kpis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_overall_kpis")

print("All Gold tables saved successfully!")


All Gold tables saved successfully!


In [0]:

# GOLD LAYER - STEP 8
# Verify all Gold tables

gold_tables = [
    "gold_daily_sales_summary",
    "gold_country_sales",
    "gold_membership_sales",
    "gold_overall_kpis"
]

for table_name in gold_tables:
    print(f"\nTable: {table_name}")
    print("Rows:", spark.table(table_name).count())
    display(spark.table(table_name))

# Check for events without matching customer information
unmatched_events = gold_base.filter(
    col("country").isNull()
)

print(
    "Events without matching customer:",
    unmatched_events.count()
)

display(unmatched_events)



Table: gold_daily_sales_summary
Rows: 17


event_date,total_events,total_purchases,unique_customers,total_revenue,average_purchase_value
2024-09-01,2,2,2,179.49,89.75
2024-09-02,2,1,2,250.00,250.00
2024-09-03,2,2,2,390.24,195.12
2024-09-04,2,1,2,65.75,65.75
2024-09-05,2,1,2,499.99,499.99
2024-09-06,2,2,2,288.95,144.48
2024-09-07,2,2,2,365.00,182.50
2024-09-08,1,1,1,39.99,39.99
2024-09-09,1,1,1,75.00,75.00
2024-09-10,1,1,1,110.00,110.00



Table: gold_country_sales
Rows: 7


country,total_customers,total_purchases,total_revenue
USA,4,8,1328.98
India,3,6,860.74
UK,2,3,565.25
Spain,1,2,519.95
Australia,2,3,395.74
Canada,2,3,215.73
South Korea,1,2,149.00



Table: gold_membership_sales
Rows: 3


membership_tier,customers_with_events,total_purchases,total_revenue,average_purchase_value
Gold,6,12,2615.18,217.93
Silver,5,9,815.48,90.61
Bronze,4,6,604.73,100.79



Table: gold_overall_kpis
Rows: 1


unique_customers,total_events,total_purchases,total_revenue,average_purchase_value
15,30,27,4035.39,149.46


Events without matching customer: 0


event_id,customer_id,event_type,event_timestamp,event_date,transaction_amount,country,membership_tier
